# LSTM Multi-Horizon Stock Prediction — Colab Training (Phase 6.5)

Run on Colab with a GPU runtime (Runtime > Change runtime type > GPU).

**Before running:** upload these 4 files from your local `data/processed/splits/`:
`train.csv`, `val.csv`, `test.csv`, `scaler.json`

**Phase 6.5 changes baked into these files:**
- Features in the split CSVs are **already scaled** by `src/prepare_dataset.py`
  (winsor 1/99% -> log1p on skewed ratios -> RobustScaler -> clip to +/-5).
  The notebook does **not** re-scale features. `scaler.json` is uploaded only
  as metadata / provenance.
- Two direction targets are trained side by side:
  `beats_median_{h}d` (all rows) and `top_tercile_{h}d` (top vs bottom third
  of each day's cross-section; the middle third is NaN and is masked out of
  that head's loss with per-sample weights).
- Regression head predicts the **vol-adjusted** return
  `fwd_return_vol_adj_{h}d` (near-unit variance), not the raw return.


In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

print('TensorFlow', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))

In [ ]:
from google.colab import files

print('Select train.csv, val.csv, test.csv, and scaler.json (all 4 at once)')
uploaded = files.upload()
assert set(uploaded.keys()) >= {'train.csv', 'val.csv', 'test.csv', 'scaler.json'}, \
    f'Missing files. Got: {list(uploaded.keys())}'

In [ ]:
# Feature / label columns — keep in sync with src/feature_engineering.py
FEATURE_COLS = [
    "log_return", "price_to_sma10", "price_to_sma20", "price_to_sma50",
    "price_to_ema20", "price_to_ema50", "price_to_ema200",
    "macd_norm", "macd_hist_norm", "rsi_14", "bb_pct_b", "bb_width",
    "volatility_10", "volatility_20", "roc_5", "roc_10", "roc_20", "roc_40",
    "atr_pct", "volume_ratio", "dist_to_high20", "dist_to_low20",
    "dist_to_high50", "dist_to_low50", "obv_zscore_20",
    "nifty_return", "nifty_volatility_20", "nifty_trend",
    "relative_strength", "sector_return",
    "vix_zscore_60", "vix_change_5d",
    "fund_roe", "fund_net_profit_growth", "fund_eps_growth", "fund_pe_ratio",
]  # 36 features, ALREADY SCALED in the split CSVs (see notebook header)

HORIZONS = [7, 30, 60, 90]
REG_LABEL_COLS     = [f"fwd_return_vol_adj_{h}d" for h in HORIZONS]  # vol-scaled return (Phase 6.5 target)
RAWRET_LABEL_COLS  = [f"fwd_return_{h}d"         for h in HORIZONS]  # raw return, kept for reference only
MEDIAN_DIR_COLS    = [f"beats_median_{h}d"       for h in HORIZONS]  # beat same-day NIFTY50 median (all rows)
TERCILE_DIR_COLS   = [f"top_tercile_{h}d"        for h in HORIZONS]  # top vs bottom third; NaN = middle third
ALL_LABEL_COLS = REG_LABEL_COLS + RAWRET_LABEL_COLS + MEDIAN_DIR_COLS + TERCILE_DIR_COLS

LOOKBACK = 60
STRIDE = 5  # stride=1 => 98%+ overlap between neighbouring windows -> instant train-noise memorisation

train = pd.read_csv('train.csv', parse_dates=['Date'])
val   = pd.read_csv('val.csv',   parse_dates=['Date'])
test  = pd.read_csv('test.csv',  parse_dates=['Date'])
print('train', train.shape, 'val', val.shape, 'test', test.shape)
print('feature abs-max (should be <= 5):',
      float(np.abs(train[FEATURE_COLS].to_numpy()).max()))


In [ ]:
def make_sequences(df, feature_cols, label_cols, lookback=LOOKBACK, stride=STRIDE):
    """Per-stock sliding windows — mirrors src/dataset.py. A window never
    mixes rows from two different companies. stride>1 spaces windows out
    to reduce the massive overlap between adjacent stride=1 windows."""
    X_parts, y_parts, meta_parts = [], [], []
    for symbol, g in df.groupby('Symbol', sort=False):
        g = g.sort_values('Date').reset_index(drop=True)
        if len(g) < lookback:
            continue
        feats = g[feature_cols].to_numpy(dtype=np.float32)
        labels = g[label_cols].to_numpy(dtype=np.float32)
        windows = np.lib.stride_tricks.sliding_window_view(feats, lookback, axis=0)
        windows = windows.transpose(0, 2, 1)
        windows = windows[::stride]
        X_parts.append(windows)
        y_parts.append(labels[lookback - 1::stride])
        meta_parts.append(g.loc[lookback - 1::stride, ['Date', 'Symbol']].reset_index(drop=True))
    X = np.concatenate(X_parts, axis=0)
    y = np.concatenate(y_parts, axis=0)
    meta = pd.concat(meta_parts, ignore_index=True)
    return X, y, meta

X_train, y_train_raw, meta_train = make_sequences(train, FEATURE_COLS, ALL_LABEL_COLS)
X_val, y_val_raw, meta_val = make_sequences(val, FEATURE_COLS, ALL_LABEL_COLS)
X_test, y_test_raw, meta_test = make_sequences(test, FEATURE_COLS, ALL_LABEL_COLS)

print('X_train', X_train.shape, 'X_val', X_val.shape, 'X_test', X_test.shape)

In [ ]:
label_idx = {name: i for i, name in enumerate(ALL_LABEL_COLS)}
reg_names = [f'ret_{h}d' for h in HORIZONS]
med_names = [f'med_{h}d' for h in HORIZONS]
ter_names = [f'ter_{h}d' for h in HORIZONS]
dir_names = med_names + ter_names

def pick(y_raw, source_cols):
    return np.stack([y_raw[:, label_idx[c]] for c in source_cols], axis=1).astype('float32')

# --- Regression target: vol-adjusted forward return -------------------------
y_train_reg_raw = pick(y_train_raw, REG_LABEL_COLS)
y_val_reg_raw   = pick(y_val_raw,   REG_LABEL_COLS)
y_test_reg_raw  = pick(y_test_raw,  REG_LABEL_COLS)

reg_mean = np.nanmean(y_train_reg_raw, axis=0)
reg_std  = np.nanstd(y_train_reg_raw, axis=0)
scale_reg   = lambda y: (y - reg_mean) / reg_std
unscale_reg = lambda y: y * reg_std + reg_mean
y_train_reg = scale_reg(y_train_reg_raw)
y_val_reg   = scale_reg(y_val_reg_raw)

# --- Direction targets: beats_median (all rows) + top_tercile (tails only) --
y_train_med = pick(y_train_raw, MEDIAN_DIR_COLS)
y_val_med   = pick(y_val_raw,   MEDIAN_DIR_COLS)
y_test_med  = pick(y_test_raw,  MEDIAN_DIR_COLS)
y_train_ter = pick(y_train_raw, TERCILE_DIR_COLS)
y_val_ter   = pick(y_val_raw,   TERCILE_DIR_COLS)
y_test_ter  = pick(y_test_raw,  TERCILE_DIR_COLS)

def to_dict(reg, med, ter):
    d = {n: reg[:, i] for i, n in enumerate(reg_names)}
    d.update({n: med[:, i] for i, n in enumerate(med_names)})
    d.update({n: ter[:, i] for i, n in enumerate(ter_names)})
    return d

def weights_dict(reg, med, ter):
    # per-sample weight = 1.0 where the label is finite, else 0.0
    # (masks the tercile middle third and any stray NaN target)
    d = {n: np.isfinite(reg[:, i]).astype('float32') for i, n in enumerate(reg_names)}
    d.update({n: np.isfinite(med[:, i]).astype('float32') for i, n in enumerate(med_names)})
    d.update({n: np.isfinite(ter[:, i]).astype('float32') for i, n in enumerate(ter_names)})
    return d

sw_train_dict = weights_dict(y_train_reg, y_train_med, y_train_ter)
sw_val_dict   = weights_dict(y_val_reg,   y_val_med,   y_val_ter)

# NaN * weight(0) is still NaN in the loss, so zero the labels AFTER weighting
clean = lambda a: np.nan_to_num(a, nan=0.0)
y_train_dict = to_dict(clean(y_train_reg), clean(y_train_med), clean(y_train_ter))
y_val_dict   = to_dict(clean(y_val_reg),   clean(y_val_med),   clean(y_val_ter))

print('tercile mask kept-fraction (train):',
      {n: round(float(sw_train_dict[n].mean()), 3) for n in ter_names})


---
## 🔵 STEP 1 — Build the LSTM model
Run the cell directly below this heading first (it builds and compiles `model`), then run the training cell right after it. If you ever see `NameError: name 'model' is not defined`, come back and re-run **this next cell**, not the training cell.

In [ ]:
from tensorflow.keras import regularizers

n_features = len(FEATURE_COLS)
N_ENSEMBLE = 5  # Gu, Kelly & Xiu (2020) style: average several independently-initialised nets

def build_lstm(seed):
    tf.random.set_seed(seed)
    l2 = regularizers.l2(1e-4)

    inputs = keras.Input(shape=(LOOKBACK, n_features), name='sequence')
    x = layers.LSTM(32, kernel_regularizer=l2, recurrent_dropout=0.2)(inputs)
    x = layers.Dropout(0.3)(x)
    trunk = layers.Dense(16, activation='relu', kernel_regularizer=l2)(x)

    reg_outputs = {n: layers.Dense(1, name=n)(trunk) for n in reg_names}
    dir_outputs = {n: layers.Dense(1, activation='sigmoid', name=n)(trunk) for n in dir_names}
    model = keras.Model(inputs=inputs, outputs={**reg_outputs, **dir_outputs})

    losses = {n: keras.losses.Huber() for n in reg_names}
    losses.update({n: 'binary_crossentropy' for n in dir_names})
    loss_weights = {n: 0.3 for n in reg_names}
    loss_weights.update({n: 1.0 for n in dir_names})
    # weighted_metrics (not metrics) so the per-sample mask is applied to
    # accuracy/MAE too -> val_ter_*_accuracy ignores the masked middle third
    weighted_metrics = {n: ['mae'] for n in reg_names}
    weighted_metrics.update({n: ['accuracy'] for n in dir_names})

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=3e-4),
        loss=losses, loss_weights=loss_weights, weighted_metrics=weighted_metrics,
    )
    return model

build_lstm(0).summary()


## 🔵 STEP 2 — Train the LSTM
Run this next. It uses `model` from Step 1 above.

In [ ]:
ensemble_models, ensemble_histories = [], []

for seed in range(N_ENSEMBLE):
    print(f'=== Training ensemble member {seed + 1}/{N_ENSEMBLE} (seed={seed}) ===')
    m = build_lstm(seed)
    callbacks = [
        keras.callbacks.EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=6, min_lr=1e-6),
    ]
    hist = m.fit(
        X_train, y_train_dict,
        sample_weight=sw_train_dict,
        validation_data=(X_val, y_val_dict, sw_val_dict),
        epochs=200, batch_size=256, callbacks=callbacks, verbose=1,
    )
    best_epoch = int(np.argmin(hist.history['val_loss']))
    print(f'--- member {seed}: stopped epoch {len(hist.history["loss"])}, '
          f'best val_loss={min(hist.history["val_loss"]):.4f} (epoch {best_epoch}) ---\n')
    ensemble_models.append(m)
    ensemble_histories.append(hist)

print(f'\nTrained {len(ensemble_models)} models.')


In [ ]:
plt.figure(figsize=(8, 4))
for i, hist in enumerate(ensemble_histories):
    plt.plot(hist.history['loss'], color=f'C{i}', alpha=0.4, label=f'member {i} train')
    plt.plot(hist.history['val_loss'], color=f'C{i}', linestyle='--', label=f'member {i} val')
plt.xlabel('epoch'); plt.ylabel('loss'); plt.legend(fontsize=7, ncol=2)
plt.title('Training curves, all ensemble members (spread = run-to-run instability)')
plt.show()

plt.figure(figsize=(8, 4))
for i, hist in enumerate(ensemble_histories):
    plt.plot(hist.history['val_med_30d_accuracy'], color=f'C{i}', label=f'member {i} beats_median')
    plt.plot(hist.history['val_ter_60d_accuracy'], color=f'C{i}', linestyle='--', label=f'member {i} tercile60')
plt.axhline(0.5, color='gray', linestyle=':', label='coin flip')
plt.xlabel('epoch'); plt.ylabel('val accuracy'); plt.legend(fontsize=7, ncol=2)
plt.title('val direction accuracy across ensemble members')
plt.show()


In [ ]:
# Ensemble = average of all members' predictions on the test set.
all_reg, all_med, all_ter = [], [], []
for m in ensemble_models:
    p = m.predict(X_test, batch_size=512, verbose=0)
    all_reg.append(np.stack([p[n].ravel() for n in reg_names], axis=1))
    all_med.append(np.stack([p[n].ravel() for n in med_names], axis=1))
    all_ter.append(np.stack([p[n].ravel() for n in ter_names], axis=1))

reg_preds_real = unscale_reg(np.mean(all_reg, axis=0))   # vol-adjusted return units
med_probs = np.mean(all_med, axis=0)
ter_probs = np.mean(all_ter, axis=0)

rows = []
for i, h in enumerate(HORIZONS):
    yt = y_test_reg_raw[:, i]; fin = np.isfinite(yt)
    mae       = np.mean(np.abs(yt[fin] - reg_preds_real[fin, i]))
    naive_mae = np.mean(np.abs(yt[fin] - reg_mean[i]))

    ym = y_test_med[:, i].astype(int)
    med_acc  = np.mean((med_probs[:, i] > 0.5).astype(int) == ym)
    med_base = max(ym.mean(), 1 - ym.mean())

    yter = y_test_ter[:, i]; k = np.isfinite(yter)
    ter_acc  = np.mean((ter_probs[k, i] > 0.5).astype(int) == yter[k].astype(int))
    ter_base = max(yter[k].mean(), 1 - yter[k].mean())

    rows.append({'h': f'{h}d',
                 'reg_MAE': round(mae, 4), 'reg_naiveMAE': round(naive_mae, 4),
                 'median_acc': round(med_acc, 4), 'median_base': round(med_base, 4),
                 'tercile_acc': round(ter_acc, 4), 'tercile_base': round(ter_base, 4)})

results_df = pd.DataFrame(rows)
print(results_df.to_string(index=False))

print('\nIndividual member tercile_60d accuracy (kept rows only):')
k60 = np.isfinite(y_test_ter[:, 2])
for i, probs in enumerate(all_ter):
    acc = np.mean((probs[k60, 2] > 0.5).astype(int) == y_test_ter[k60, 2].astype(int))
    print(f'  member {i}: {acc:.4f}')


In [ ]:
# Save all ensemble members + label-scaling stats + head map, zip, download.
import shutil, os, json as _json

os.makedirs('lstm_ensemble', exist_ok=True)
for i, m in enumerate(ensemble_models):
    m.save(f'lstm_ensemble/member_{i}.keras')
np.savez('lstm_ensemble/label_scaler.npz',
         mean=reg_mean, std=reg_std, reg_cols=np.array(REG_LABEL_COLS))
_json.dump(
    {'reg_names': reg_names, 'med_names': med_names, 'ter_names': ter_names,
     'HORIZONS': HORIZONS, 'LOOKBACK': LOOKBACK, 'STRIDE': STRIDE,
     'features_prescaled': True,
     'note': 'features are pre-scaled by src/prepare_dataset.apply_saved_scaling; '
             'regression heads predict standardized fwd_return_vol_adj_{h}d'},
    open('lstm_ensemble/heads.json', 'w'), indent=2)

shutil.make_archive('lstm_ensemble', 'zip', 'lstm_ensemble')
files.download('lstm_ensemble.zip')


## Alternative: Dense (MLP) network + hyperparameter search

The verified signal (plain logistic regression beating naive) came from a
**linear** model on just the *current day's* 36 features — no sequence
history. This section tests whether a small feedforward net on that same
flat vector does any better than the LSTM. Same targets, same masking.
Uses `X_train`/`y_train_dict`/`sw_train_dict` already in memory.


In [ ]:
from tensorflow.keras import regularizers

X_train_flat = X_train[:, -1, :]   # just "today's" (already-scaled) features
X_val_flat   = X_val[:, -1, :]
X_test_flat  = X_test[:, -1, :]

def build_dense_model(hidden_units, dropout, l2_reg, lr):
    l2 = regularizers.l2(l2_reg)
    inputs = keras.Input(shape=(len(FEATURE_COLS),), name='features')
    x = inputs
    for units in hidden_units:
        x = layers.Dense(units, activation='relu', kernel_regularizer=l2)(x)
        x = layers.Dropout(dropout)(x)
    reg_outputs = {n: layers.Dense(1, name=n)(x) for n in reg_names}
    dir_outputs = {n: layers.Dense(1, activation='sigmoid', name=n)(x) for n in dir_names}
    model = keras.Model(inputs=inputs, outputs={**reg_outputs, **dir_outputs})
    losses = {n: keras.losses.Huber() for n in reg_names}
    losses.update({n: 'binary_crossentropy' for n in dir_names})
    loss_weights = {n: 0.3 for n in reg_names}
    loss_weights.update({n: 1.0 for n in dir_names})
    weighted_metrics = {n: ['accuracy'] for n in dir_names}
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                  loss=losses, loss_weights=loss_weights, weighted_metrics=weighted_metrics)
    return model

configs = [
    {'hidden_units': [64, 32],      'dropout': 0.3, 'l2_reg': 1e-4, 'lr': 1e-3},
    {'hidden_units': [64, 32],      'dropout': 0.5, 'l2_reg': 1e-3, 'lr': 1e-3},
    {'hidden_units': [32, 16],      'dropout': 0.4, 'l2_reg': 1e-4, 'lr': 5e-4},
    {'hidden_units': [128, 64, 32], 'dropout': 0.4, 'l2_reg': 1e-3, 'lr': 1e-3},
    {'hidden_units': [32],          'dropout': 0.2, 'l2_reg': 1e-3, 'lr': 1e-3},
]

search_results = []
for i, cfg in enumerate(configs):
    print(f'--- Config {i}: {cfg} ---')
    m = build_dense_model(**cfg)
    es = keras.callbacks.EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True)
    hist = m.fit(
        X_train_flat, y_train_dict, sample_weight=sw_train_dict,
        validation_data=(X_val_flat, y_val_dict, sw_val_dict),
        epochs=60, batch_size=256, callbacks=[es], verbose=0,
    )
    best_val_loss = min(hist.history['val_loss'])
    val_accs = {n: round(max(hist.history[f'val_{n}_accuracy']), 4) for n in dir_names}
    print(f'  best val_loss={best_val_loss:.4f}  val_accs={val_accs}')
    search_results.append({'config': cfg, 'val_loss': best_val_loss, 'model': m})

best = min(search_results, key=lambda r: r['val_loss'])
print('\nBest config (lowest val_loss):', best['config'])
dense_model = best['model']


In [ ]:
# Evaluate the best dense model on the test set - same layout as the LSTM cell.
p = dense_model.predict(X_test_flat, batch_size=512)
reg_preds_real = unscale_reg(np.stack([p[n].ravel() for n in reg_names], axis=1))
med_probs = np.stack([p[n].ravel() for n in med_names], axis=1)
ter_probs = np.stack([p[n].ravel() for n in ter_names], axis=1)

rows = []
for i, h in enumerate(HORIZONS):
    yt = y_test_reg_raw[:, i]; fin = np.isfinite(yt)
    mae       = np.mean(np.abs(yt[fin] - reg_preds_real[fin, i]))
    naive_mae = np.mean(np.abs(yt[fin] - reg_mean[i]))
    ym = y_test_med[:, i].astype(int)
    med_acc  = np.mean((med_probs[:, i] > 0.5).astype(int) == ym)
    med_base = max(ym.mean(), 1 - ym.mean())
    yter = y_test_ter[:, i]; k = np.isfinite(yter)
    ter_acc  = np.mean((ter_probs[k, i] > 0.5).astype(int) == yter[k].astype(int))
    ter_base = max(yter[k].mean(), 1 - yter[k].mean())
    rows.append({'h': f'{h}d', 'reg_MAE': round(mae, 4), 'reg_naiveMAE': round(naive_mae, 4),
                 'median_acc': round(med_acc, 4), 'median_base': round(med_base, 4),
                 'tercile_acc': round(ter_acc, 4), 'tercile_base': round(ter_base, 4)})
print(pd.DataFrame(rows).to_string(index=False))
